In [1]:
import pandas as pd
from datetime import datetime

# ============================================================
# 2025年度 社会保険料率（最新版）
# ============================================================

# 厚生年金保険料率（全国一律・労使折半）
KOSEI_NENKIN_RATE = 0.183  # 18.3%（本人負担9.15%）

# 雇用保険料率（2025年度）
KOYO_HOKEN_RATE = {
    "一般": 0.006,       # 本人負担0.6%
    "農林水産・清酒製造": 0.007,
    "建設": 0.007
}

# 健康保険料率（協会けんぽ・都道府県別 2025年度）
# ※本人負担は料率の半分
KENKO_HOKEN_RATE = {
    "北海道": 0.1063, "青森": 0.1031, "岩手": 0.1004,
    "宮城": 0.1025,   "秋田": 0.1035, "山形": 0.1012,
    "福島": 0.1017,   "茨城": 0.1010, "栃木": 0.1012,
    "群馬": 0.1015,   "埼玉": 0.1000, "千葉": 0.1010,
    "東京": 0.1000,   "神奈川": 0.1002,"新潟": 0.0978,
    "富山": 0.1011,   "石川": 0.1024, "福井": 0.1019,
    "山梨": 0.1025,   "長野": 0.0980, "静岡": 0.0980,
    "愛知": 0.1000,   "三重": 0.1001, "滋賀": 0.1000,
    "京都": 0.1018,   "大阪": 0.1029, "兵庫": 0.1026,
    "奈良": 0.1016,   "和歌山": 0.1025,"鳥取": 0.1013,
    "島根": 0.1033,   "岡山": 0.1016, "広島": 0.1008,
    "山口": 0.1021,   "徳島": 0.1058, "香川": 0.1023,
    "愛媛": 0.1023,   "高知": 0.1090, "福岡": 0.1036,
    "佐賀": 0.1055,   "長崎": 0.1040, "熊本": 0.1022,
    "大分": 0.1042,   "宮崎": 0.1045, "鹿児島": 0.1043,
    "沖縄": 0.1018
}

# 介護保険料率（2025年度・全国一律）
KAIGO_HOKEN_RATE = 0.018  # 1.8%（本人負担0.9%）

print("✅ 2025年度 保険料率の設定完了")
print(f"　厚生年金：{KOSEI_NENKIN_RATE*100:.1f}%（本人負担{KOSEI_NENKIN_RATE/2*100:.2f}%）")
print(f"　健康保険（東京）：{KENKO_HOKEN_RATE['東京']*100:.2f}%（本人負担{KENKO_HOKEN_RATE['東京']/2*100:.3f}%）")
print(f"　介護保険：{KAIGO_HOKEN_RATE*100:.1f}%（本人負担{KAIGO_HOKEN_RATE/2*100:.2f}%）")
print(f"　雇用保険（一般）：{KOYO_HOKEN_RATE['一般']*100:.1f}%")

✅ 2025年度 保険料率の設定完了
　厚生年金：18.3%（本人負担9.15%）
　健康保険（東京）：10.00%（本人負担5.000%）
　介護保険：1.8%（本人負担0.90%）
　雇用保険（一般）：0.6%


In [2]:
# ============================================================
# 従業員データの作成
# ============================================================
employees = pd.DataFrame({
    "氏名":     ["田中 太郎", "鈴木 花子", "佐藤 次郎", "山田 美咲", "伊藤 健一"],
    "年齢":     [45, 38, 52, 29, 41],
    "都道府県": ["東京", "神奈川", "大阪", "千葉", "東京"],
    "標準報酬月額": [300000, 250000, 400000, 220000, 350000],
    "業種":     ["一般", "一般", "一般", "一般", "一般"]
})

# ============================================================
# 保険料計算関数
# ============================================================
def calc_shakai_hoken(row):
    kyuyo = row["標準報酬月額"]
    age   = row["年齢"]
    pref  = row["都道府県"]
    gyoshu = row["業種"]

    # 健康保険料（本人負担：料率の半分）
    kenko_rate = KENKO_HOKEN_RATE.get(pref, 0.1000)
    kenko = round(kyuyo * kenko_rate / 2)

    # 介護保険料（40歳以上のみ）
    kaigo = round(kyuyo * KAIGO_HOKEN_RATE / 2) if age >= 40 else 0

    # 厚生年金保険料
    nenkin = round(kyuyo * KOSEI_NENKIN_RATE / 2)

    # 雇用保険料
    koyo_rate = KOYO_HOKEN_RATE.get(gyoshu, 0.006)
    koyo = round(kyuyo * koyo_rate)

    # 合計控除額
    total = kenko + kaigo + nenkin + koyo

    return pd.Series({
        "健康保険料": kenko,
        "介護保険料": kaigo,
        "厚生年金料": nenkin,
        "雇用保険料": koyo,
        "社保控除合計": total,
        "手取り概算": kyuyo - total
    })

# 計算実行
result = employees.join(employees.apply(calc_shakai_hoken, axis=1))

print("✅ 社会保険料計算完了")
print(f"対象者：{len(result)}名")
result

✅ 社会保険料計算完了
対象者：5名


,氏名,年齢,都道府県,標準報酬月額,業種,健康保険料,介護保険料,厚生年金料,雇用保険料,社保控除合計,手取り概算
0,田中 太郎,45,東京,300000,一般,15000,2700,27450,1800,46950,253050
1,鈴木 花子,38,神奈川,250000,一般,12525,0,22875,1500,36900,213100
2,佐藤 次郎,52,大阪,400000,一般,20580,3600,36600,2400,63180,336820
3,山田 美咲,29,千葉,220000,一般,11110,0,20130,1320,32560,187440
4,伊藤 健一,41,東京,350000,一般,17500,3150,32025,2100,54775,295225


In [3]:
import math
from decimal import Decimal, ROUND_HALF_UP

def round_shakai(yen):
    """社会保険の正しい四捨五入（50銭以上切り上げ）"""
    return int(Decimal(str(yen)).quantize(Decimal('1'), rounding=ROUND_HALF_UP))

def floor_shakai(yen):
    """雇用保険の正しい切り捨て"""
    return math.floor(yen)

def calc_shakai_hoken_v2(row):
    kyuyo  = row["標準報酬月額"]
    age    = row["年齢"]
    pref   = row["都道府県"]
    gyoshu = row["業種"]

    # 健康保険料（四捨五入）
    kenko_rate = KENKO_HOKEN_RATE.get(pref, 0.1000)
    kenko = round_shakai(kyuyo * kenko_rate / 2)

    # 介護保険料（四捨五入・40歳以上のみ）
    kaigo = round_shakai(kyuyo * KAIGO_HOKEN_RATE / 2) if age >= 40 else 0

    # 厚生年金保険料（四捨五入）
    nenkin = round_shakai(kyuyo * KOSEI_NENKIN_RATE / 2)

    # 雇用保険料（切り捨て）
    koyo_rate = KOYO_HOKEN_RATE.get(gyoshu, 0.006)
    koyo = floor_shakai(kyuyo * koyo_rate)

    total = kenko + kaigo + nenkin + koyo

    return pd.Series({
        "健康保険料": kenko,
        "介護保険料": kaigo,
        "厚生年金料": nenkin,
        "雇用保険料": koyo,
        "社保控除合計": total,
        "手取り概算": kyuyo - total
    })

# 修正版で再計算
result_v2 = employees.join(employees.apply(calc_shakai_hoken_v2, axis=1))

print("✅ 端数処理修正版で再計算完了")
result_v2

✅ 端数処理修正版で再計算完了


,氏名,年齢,都道府県,標準報酬月額,業種,健康保険料,介護保険料,厚生年金料,雇用保険料,社保控除合計,手取り概算
0,田中 太郎,45,東京,300000,一般,15000,2700,27450,1800,46950,253050
1,鈴木 花子,38,神奈川,250000,一般,12525,0,22875,1500,36900,213100
2,佐藤 次郎,52,大阪,400000,一般,20580,3600,36600,2400,63180,336820
3,山田 美咲,29,千葉,220000,一般,11110,0,20130,1320,32560,187440
4,伊藤 健一,41,東京,350000,一般,17500,3150,32025,2100,54775,295225


In [4]:
print(result == result_v2)

     氏名    年齢  都道府県  標準報酬月額    業種  健康保険料  介護保険料  厚生年金料  雇用保険料  社保控除合計  手取り概算
0  True  True  True    True  True   True   True   True   True    True   True
1  True  True  True    True  True   True   True   True   True    True   True
2  True  True  True    True  True   True   True   True   True    True   True
3  True  True  True    True  True   True   True   True   True    True   True
4  True  True  True    True  True   True   True   True   True    True   True


In [6]:
from google.colab import drive
drive.mount('/content/drive')

from datetime import datetime

now = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = f"/content/drive/MyDrive/Colab Notebooks/社会保険料計算_{now}.xlsx"

result_v2.to_excel(output_path, index=False, sheet_name="社保計算")

print("✅ Excel書き出し完了")
print(f"ファイル名：社会保険料計算_{now}.xlsx")
print(f"対象者：{len(result_v2)}名")
print(f"社保控除合計（全員）：{result_v2['社保控除合計'].sum():,}円")

Mounted at /content/drive
✅ Excel書き出し完了
ファイル名：社会保険料計算_20260408_224213.xlsx
対象者：5名
社保控除合計（全員）：234,365円
